In [4]:
from model import LidarCenterNet
import torch

transfuser_model_path = "../model_ckpt/models_2022/latentTF/model_seed1_41.pth"
diffusion_drive_path  = '../model_ckpt/diffusiondrive/model_10.pth'

device = 'gpu' if torch.cuda.is_available() else "cpu"

tf_model_ckpt = torch.load(transfuser_model_path, map_location='cpu')
dd_model_ckpt = torch.load(diffusion_drive_path, map_location='cpu')


state_tf = tf_model_ckpt['state_dict'] if 'state_dict' in tf_model_ckpt else tf_model_ckpt
state_diff = dd_model_ckpt['state_dict'] if 'state_dict' in dd_model_ckpt else dd_model_ckpt

In [ ]:
print("=== state_tf ===")
for k, v in state_tf.items():
    print(f"{k:<60} {tuple(v.shape)}")


=== state_tf ===
module._model.image_encoder.features.stem.conv.weight        (32, 3, 3, 3)
module._model.image_encoder.features.stem.bn.weight          (32,)
module._model.image_encoder.features.stem.bn.bias            (32,)
module._model.image_encoder.features.stem.bn.running_mean    (32,)
module._model.image_encoder.features.stem.bn.running_var     (32,)
module._model.image_encoder.features.stem.bn.num_batches_tracked ()
module._model.image_encoder.features.s1.b1.conv1.conv.weight (72, 32, 1, 1)
module._model.image_encoder.features.s1.b1.conv1.bn.weight   (72,)
module._model.image_encoder.features.s1.b1.conv1.bn.bias     (72,)
module._model.image_encoder.features.s1.b1.conv1.bn.running_mean (72,)
module._model.image_encoder.features.s1.b1.conv1.bn.running_var (72,)
module._model.image_encoder.features.s1.b1.conv1.bn.num_batches_tracked ()
module._model.image_encoder.features.s1.b1.conv2.conv.weight (72, 24, 3, 3)
module._model.image_encoder.features.s1.b1.conv2.bn.weight   (72,)
mod

In [7]:

print("\n=== state_diff ===")
for k, v in state_diff.items():
    print(f"{k:<60} {tuple(v.shape)}")


=== state_diff ===
_bev_downscale.weight                                        (256, 512, 1, 1)
_bev_downscale.bias                                          (256,)
_status_encoding.weight                                      (256, 6)
_status_encoding.bias                                        (256,)
_keyval_embedding.weight                                     (65, 256)
_query_embedding.weight                                      (31, 256)
bev_proj.0.weight                                            (256, 320)
bev_proj.0.bias                                              (256,)
bev_proj.2.weight                                            (256,)
bev_proj.2.bias                                              (256,)
_tf_decoder.layers.0.self_attn.in_proj_weight                (768, 256)
_tf_decoder.layers.0.self_attn.in_proj_bias                  (768,)
_tf_decoder.layers.0.self_attn.out_proj.weight               (256, 256)
_tf_decoder.layers.0.self_attn.out_proj.bias                 (256,

In [94]:

# start from tf
merged = {}

for k, v in state_tf.items():
    # keep everything up to join.*
    if k.startswith("module.decoder") or k.startswith("module.output"):
        # skip decoder from tf
        continue
    elif k.startswith("module."):
        new_k = k.replace("module.", "", 1)  # only remove the first "module."
        merged[new_k] = v
  
 
    else:
        # keep other layers (encoder etc.)
        merged[k] = v



# now bring in path_out from diff
for k, v in state_diff.items():
    if k.startswith('_tf_decoder') or k.startswith('bev_proj') or k.startswith('_query_embedding') or k.startswith("_keyval_embedding") or k.startswith("path_out") or k.startswith("_bev_downscale") or k.startswith('_status_encoding'):
        merged[k] = v
        print(f"Added {k} from state_diff: {tuple(v.shape)}")

# save merged
torch.save({"state_dict": merged}, "dd.pth")

Added _bev_downscale.weight from state_diff: (256, 512, 1, 1)
Added _bev_downscale.bias from state_diff: (256,)
Added _status_encoding.weight from state_diff: (256, 6)
Added _status_encoding.bias from state_diff: (256,)
Added _keyval_embedding.weight from state_diff: (65, 256)
Added _query_embedding.weight from state_diff: (31, 256)
Added bev_proj.0.weight from state_diff: (256, 320)
Added bev_proj.0.bias from state_diff: (256,)
Added bev_proj.2.weight from state_diff: (256,)
Added bev_proj.2.bias from state_diff: (256,)
Added _tf_decoder.layers.0.self_attn.in_proj_weight from state_diff: (768, 256)
Added _tf_decoder.layers.0.self_attn.in_proj_bias from state_diff: (768,)
Added _tf_decoder.layers.0.self_attn.out_proj.weight from state_diff: (256, 256)
Added _tf_decoder.layers.0.self_attn.out_proj.bias from state_diff: (256,)
Added _tf_decoder.layers.0.multihead_attn.in_proj_weight from state_diff: (768, 256)
Added _tf_decoder.layers.0.multihead_attn.in_proj_bias from state_diff: (768,)

In [95]:
new_model_path = 'dd.pth'
new_model_ckpt = torch.load(new_model_path, map_location='cpu')

state_new = new_model_ckpt['state_dict'] if 'state_dict' in new_model_ckpt else new_model_ckpt
# print("=== state_tf ===")
# for k, v in state_new.items():
#     print(f"{k:<60} {tuple(v.shape)}")


In [104]:
import os
import json
import torch
from model import LidarCenterNet
from config import GlobalConfig
import numpy as np
from PIL import Image
from data import lidar_to_histogram_features, draw_target_point
path_to_conf_file = "/home/fypits25/Documents/tfddcarla/model_ckpt/diffusiondrive"
model_file = "model_10.pth"

args_file = open(os.path.join(path_to_conf_file, 'args.txt'), 'r')
args = json.load(args_file)
args_file.close()

config = GlobalConfig(setting='eval')
if ('sync_batch_norm' in args):
    config.sync_batch_norm = bool(args['sync_batch_norm'])
if ('use_point_pillars' in args):
    config.use_point_pillars = args['use_point_pillars']
if ('n_layer' in args):
    config.n_layer = args['n_layer']
if ('use_target_point_image' in args):
    config.use_target_point_image = bool(args['use_target_point_image'])
if ('use_velocity' in args):
    use_velocity = bool(args['use_velocity'])
else:
    use_velocity = True

if ('image_architecture' in args):
    image_architecture = args['image_architecture']
else:
    image_architecture = 'resnet34'

if ('lidar_architecture' in args):
    lidar_architecture = args['lidar_architecture']
else:
    lidar_architecture = 'resnet18'

if ('backbone' in args):
    backbone = args['backbone']  # Options 'geometric_fusion', 'transFuser', 'late_fusion', 'latentTF'
else:
    backbone = 'transFuser'  # Options 'geometric_fusion', 'transFuser', 'late_fusion', 'latentTF'

# Load model files
import sys, os

# Suppress prints
stdout_backup = sys.stdout
sys.stdout = open(os.devnull, 'w')

net = LidarCenterNet(config, 'cuda', backbone, "diffusiondrive", image_architecture, lidar_architecture, use_velocity)

# Re-enable prints
sys.stdout.close()
sys.stdout = stdout_backup

In [105]:
net.load_state_dict(merged)


<All keys matched successfully>

In [103]:

# start from tf
merged = {}

for k, v in state_tf.items():
    # keep everything up to join.*
    if k.startswith("module.decoder") or k.startswith("module.output") or k.startswith('module._model.lidar_encoder._model.stem.conv.weight'):
        # skip decoder from tf
        continue
    elif k.startswith("module."):
        new_k = k.replace("module.", "", 1)  # only remove the first "module."
        merged[new_k] = v
  
 
    else:
        # keep other layers (encoder etc.)
        merged[k] = v



# now bring in path_out from diff
for k, v in state_diff.items():
    if k.startswith('_tf_decoder') or k.startswith('bev_proj') or k.startswith('_query_embedding') or k.startswith("_keyval_embedding") or k.startswith("path_out") or k.startswith("_bev_downscale") or k.startswith('_status_encoding'):
        merged[k] = v
        print(f"Added {k} from state_diff: {tuple(v.shape)}")

# save merged
torch.save({"state_dict": merged}, "dd.pth")

Added _bev_downscale.weight from state_diff: (256, 512, 1, 1)
Added _bev_downscale.bias from state_diff: (256,)
Added _status_encoding.weight from state_diff: (256, 6)
Added _status_encoding.bias from state_diff: (256,)
Added _keyval_embedding.weight from state_diff: (65, 256)
Added _query_embedding.weight from state_diff: (31, 256)
Added bev_proj.0.weight from state_diff: (256, 320)
Added bev_proj.0.bias from state_diff: (256,)
Added bev_proj.2.weight from state_diff: (256,)
Added bev_proj.2.bias from state_diff: (256,)
Added _tf_decoder.layers.0.self_attn.in_proj_weight from state_diff: (768, 256)
Added _tf_decoder.layers.0.self_attn.in_proj_bias from state_diff: (768,)
Added _tf_decoder.layers.0.self_attn.out_proj.weight from state_diff: (256, 256)
Added _tf_decoder.layers.0.self_attn.out_proj.bias from state_diff: (256,)
Added _tf_decoder.layers.0.multihead_attn.in_proj_weight from state_diff: (768, 256)
Added _tf_decoder.layers.0.multihead_attn.in_proj_bias from state_diff: (768,)